# 0. Import required packages

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
import math
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
import joblib
import numpy as np


## 1. Create Baseline Model

## 1.1 Create a table with the baseline column

In [0]:
%sql
CREATE OR REPLACE TABLE taxi_with_baseline AS
WITH baseline AS (
    SELECT
        taxi_type,
        pickup_borough,
        dropoff_borough,
        DATE_FORMAT(pickup_datetime, 'yyyy-MM') AS year_month,
        YEAR(pickup_datetime) AS year,
        MONTH(pickup_datetime) AS month,
        DAYOFWEEK(pickup_datetime) AS day_of_week,
        HOUR(pickup_datetime) AS hour_of_day,
        ROUND(AVG(total_amount), 2) AS avg_amount_per_trip
    FROM cleaned_taxi_data
    GROUP BY
        taxi_type,
        pickup_borough,
        dropoff_borough,
        DATE_FORMAT(pickup_datetime, 'yyyy-MM'),
        YEAR(pickup_datetime),
        MONTH(pickup_datetime),
        DAYOFWEEK(pickup_datetime),
        HOUR(pickup_datetime)
)
SELECT
    c.*,
    b.year,
    b.month,
    b.hour_of_day,
    b.avg_amount_per_trip
FROM cleaned_taxi_data c
JOIN baseline b
  ON c.taxi_type = b.taxi_type
 AND c.pickup_borough = b.pickup_borough
 AND c.dropoff_borough = b.dropoff_borough
 AND DATE_FORMAT(c.pickup_datetime, 'yyyy-MM') = b.year_month  
 AND YEAR(c.pickup_datetime) = b.year
 AND MONTH(c.pickup_datetime) = b.month
 AND DAYOFWEEK(c.pickup_datetime) = b.day_of_week
 AND HOUR(c.pickup_datetime) = b.hour_of_day;


In [0]:
df = spark.table("taxi_with_baseline")

In [0]:
# Add day_of_week feature
df = df.withColumn("day_of_week", F.date_format("pickup_datetime", "EEEE"))

In [0]:
# Create year_month column for df_train and test
df = df.withColumn("year_month", F.date_format(df["pickup_datetime"], "yyyy-MM"))

In [0]:
display(df)

## 2. Splitting the Data

In [0]:
df_test = df.filter(F.col("pickup_datetime") >= "2024-10-01")
df_trainval = df.filter(F.col("pickup_datetime") < "2024-10-01")

In [0]:

# 1% stratified sampling by year_month_index
year_month_index_values = [
    row['year_month'] for row in df_trainval.select('year_month').distinct().collect()
]

fractions = {ym: 0.01 for ym in year_month_index_values}  # 2% from each month

df_sample = df_trainval.sampleBy("year_month", fractions=fractions, seed=42) \
                       .withColumn("rand", F.rand(seed=123))

# Define train fraction
train_frac = 0.7

# Split stratified by year_month_index
df_train = df_sample.filter(F.col("rand") <= train_frac).drop("rand")
df_val   = df_sample.filter(F.col("rand") > train_frac).drop("rand")

In [0]:
# Counts
print("Train rows:", df_train.count())
print("Validation rows:", df_val.count())
print("Test rows:", df_test.count())

In [0]:
def compute_baseline_rmse(df, prediction_col="avg_amount_per_trip", label_col="total_amount"):
    # Compute squared error: (prediction - label)^2
    df_with_error = df.select(
        F.pow(F.col(prediction_col) - F.col(label_col), 2).alias("squared_error")
    )
    
    # Compute RMSE directly using Spark (sqrt of average squared error)
    rmse = df_with_error.select(F.sqrt(F.avg("squared_error")).alias("rmse")).collect()[0]["rmse"]
    
    return rmse

In [0]:
rmse_train = compute_baseline_rmse(df_train)
print(f"Train RMSE: {rmse_train}")

rmse_val = compute_baseline_rmse(df_val)
print(f"Validation RMSE: {rmse_val}")

rmse_test = compute_baseline_rmse(df_test)
print(f"Test RMSE: {rmse_test}")

## 3. Modelling

### [3.2] Test One-hot encoding using Spark & borough features

In [0]:
df_pivot = df_train.select('pickup_borough', 'dropoff_borough') \
    .withColumn('pickup_Brooklyn', F.when(F.col('pickup_borough') == 'Brooklyn', 1).otherwise(0)) \
    .withColumn('pickup_Bronx', F.when(F.col('pickup_borough') == 'Bronx', 1).otherwise(0)) \
    .withColumn('pickup_Manhattan', F.when(F.col('pickup_borough') == 'Manhattan', 1).otherwise(0)) \
    .withColumn('pickup_Queens', F.when(F.col('pickup_borough') == 'Queens', 1).otherwise(0)) \
    .withColumn('pickup_StatenIsland', F.when(F.col('pickup_borough') == 'Staten Island', 1).otherwise(0)) \
    .withColumn('pickup_EWR', F.when(F.col('pickup_borough') == 'EWR', 1).otherwise(0)) \
    .withColumn('pickup_Unknown', F.when(F.col('pickup_borough') == 'Unknown', 1).otherwise(0)) \
    .withColumn('dropoff_Brooklyn', F.when(F.col('dropoff_borough') == 'Brooklyn', 1).otherwise(0)) \
    .withColumn('dropoff_Bronx', F.when(F.col('dropoff_borough') == 'Bronx', 1).otherwise(0)) \
    .withColumn('dropoff_Manhattan', F.when(F.col('dropoff_borough') == 'Manhattan', 1).otherwise(0)) \
    .withColumn('dropoff_Queens', F.when(F.col('dropoff_borough') == 'Queens', 1).otherwise(0)) \
    .withColumn('dropoff_StatenIsland', F.when(F.col('dropoff_borough') == 'Staten Island', 1).otherwise(0)) \
    .withColumn('dropoff_EWR', F.when(F.col('dropoff_borough') == 'EWR', 1).otherwise(0)) \
    .withColumn('dropoff_Unknown', F.when(F.col('dropoff_borough') == 'Unknown', 1).otherwise(0))

In [0]:
df_pivot = df_pivot.drop('pickup_borough', 'dropoff_borough')

In [0]:
display(df_pivot)

###[3.3] Define a helper function that does the above OHE with Spark

Categories dict was created because there were some issues with the order of the one_hot_encode fucntion when trying to chunk train the data based on different months.

+ day_of_week and hour_of_day are the final categorical variables

In [0]:
cat_cols = ["day_of_week", 'hour_of_day']
categories_dict = {}

for col in cat_cols:
    categories_dict[col] = [row[0] for row in df.select(col).distinct().collect()]

print(categories_dict)

In [0]:
def one_hot_encode_spark(df, cat_cols, categories_dict, drop_first=False):
    """
    One-hot encode categorical columns in Spark using a fixed categories_dict.
    Ensures consistent columns across chunks and optionally drops the first dummy to avoid multicollinearity.

    Args:
        df: Spark DataFrame
        cat_cols: list of categorical columns to encode
        categories_dict: dict {col: [list of categories]} fixed for all chunks
        drop_first: bool, if True drops the first dummy per categorical column
        drop_original: bool, if True drops the original categorical columns
    """
    exprs = [df["*"]]  # keep all existing columns by default

    for col in cat_cols:
        categories = categories_dict[col]
        categories_to_encode = categories[1:] if drop_first else categories

        for val in categories_to_encode:
            exprs.append(
                F.when(F.col(col) == val, 1).otherwise(0).alias(f"{col}_{val}")
            )

    # Apply all encodings in one pass
    df_encoded = df.select(*exprs)
    df_encoded = df_encoded.drop(*cat_cols)

    return df_encoded

####[3.3.1] Testing the function

In [0]:
df_1 =  df_train.select('day_of_week')

In [0]:
display(df_1)

In [0]:

df_1 = one_hot_encode_spark(df_1, cat_cols,categories_dict)

In [0]:
display(df_1)

### [4.1] Linear Regression

In [0]:
# Define label and features
label_col = "total_amount"
numerical_features = ['trip_distance', 'trip_duration_hours', 'year', 'month']
categorical_features = ['day_of_week', 'hour_of_day']

In [0]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()
scaler = StandardScaler()

In [0]:
# Select features + label together in one DataFrame
cols = numerical_features + categorical_features + [label_col]
df_all = df_train.select(*cols)

# One-hot encode categorical features in Spark
df_all_encoded = one_hot_encode_spark(df_all, categorical_features, categories_dict)

# Convert to pandas
df_pd = df_all_encoded.toPandas()

# Split features and target
X_pd = df_pd.drop(columns=[label_col])
y_pd = df_pd[label_col]

# Scale numerical features
X_pd[numerical_features] = scaler.fit_transform(X_pd[numerical_features])

# Train the model
linear_model.fit(X_pd, y_pd)

In [0]:
print(linear_model.coef_)

In [0]:
 preds = linear_model.predict(X_pd)
 y_true = y_pd
 rmse = np.sqrt(mean_squared_error(y_true, preds))

 print(f'Training set RMSE: {rmse}')

In [0]:
# Select features + label together
cols_val = numerical_features + categorical_features + [label_col]
df_val_all = df_val.select(*cols_val)

# One-hot encode categorical columns
df_val_encoded = one_hot_encode_spark(df_val_all, categorical_features, categories_dict)


# Convert to pandas
df_val_pd = df_val_encoded.toPandas()

# Split features / target
X_val_pd = df_val_pd.drop(columns=[label_col])
y_val_pd = df_val_pd[label_col]

# Scale
X_val_pd[numerical_features] = scaler.transform(X_val_pd[numerical_features])

# Predict
preds_val = linear_model.predict(X_val_pd)
rmse_val = np.sqrt(mean_squared_error(y_val_pd, preds_val))
print(f"Validation set RMSE: {rmse_val:.4f}")

The model shows sign of overfitting. Add regularization parameter

In [0]:
# --- Test set RMSE / predictions ---

# Select features + label together
cols_test = numerical_features + categorical_features + [label_col]  # include label if available
df_test_all = df_test.select(*cols_test)

#  One-hot encode categorical columns
df_test_encoded = one_hot_encode_spark(df_test_all, categorical_features, categories_dict)

# Convert to pandas
df_test_pd = df_test_encoded.toPandas()

# Split features / target
X_test_pd = df_test_pd.drop(columns=[label_col])
y_test_pd = df_test_pd[label_col]  # if label available; otherwise omit

# Scale numerical features using the SAME scaler fitted on training
X_test_pd[numerical_features] = scaler.transform(X_test_pd[numerical_features])

#Predict
preds_test = linear_model.predict(X_test_pd)

rmse_test = np.sqrt(mean_squared_error(y_test_pd, preds_test))
print(f"Test set RMSE: {rmse_test:.4f}")

### [4.2] Elastic Net 

In [0]:
from sklearn.linear_model import ElasticNet

In [0]:
elastic = ElasticNet(alpha=1.5, l1_ratio=0.5, random_state=42, max_iter=1000)
scaler = StandardScaler()

In [0]:
# Select features + label together in one DataFrame
cols = numerical_features + categorical_features + [label_col]
df_all = df_train.select(*cols)

# One-hot encode categorical features in Spark
df_all_encoded = one_hot_encode_spark(df_all, categorical_features, categories_dict)

# Convert to pandas
df_pd = df_all_encoded.toPandas()

# Split features and target
X_pd = df_pd.drop(columns=[label_col])
y_pd = df_pd[label_col]

# Scale numerical features
X_pd[numerical_features] = scaler.fit_transform(X_pd[numerical_features])

# Train the model
elastic.fit(X_pd, y_pd)

In [0]:
 preds = elastic.predict(X_pd)
 y_true = y_pd
 rmse = np.sqrt(mean_squared_error(y_true, preds))

 print(f'Training set RMSE: {rmse}')

In [0]:
# Select features + label together
cols_val = numerical_features + categorical_features + [label_col]
df_val_all = df_val.select(*cols_val)

# One-hot encode categorical columns
df_val_encoded = one_hot_encode_spark(df_val_all, categorical_features, categories_dict)

# Convert to pandas
df_val_pd = df_val_encoded.toPandas()

# Split features / target
X_val_pd = df_val_pd.drop(columns=[label_col])
y_val_pd = df_val_pd[label_col]

# Scale
X_val_pd[numerical_features] = scaler.transform(X_val_pd[numerical_features])

# Predict
preds_val = elastic.predict(X_val_pd)
rmse_val = np.sqrt(mean_squared_error(y_val_pd, preds_val))
print(f"Validation set RMSE: {rmse_val:.4f}")

In [0]:
# --- Test set RMSE / predictions ---

# Select features + label together
cols_test = numerical_features + categorical_features + [label_col]  # include label if available
df_test_all = df_test.select(*cols_test)

#  One-hot encode categorical columns
df_test_encoded = one_hot_encode_spark(df_test_all, categorical_features, categories_dict)

# Convert to pandas
df_test_pd = df_test_encoded.toPandas()

# Split features / target
X_test_pd = df_test_pd.drop(columns=[label_col])
y_test_pd = df_test_pd[label_col]  # if label available; otherwise omit

# Scale numerical features using the SAME scaler fitted on training
X_test_pd[numerical_features] = scaler.transform(X_test_pd[numerical_features])

#Predict
preds_test = elastic.predict(X_test_pd)

rmse_test = np.sqrt(mean_squared_error(y_test_pd, preds_test))
print(f"Test set RMSE: {rmse_test:.4f}")

###[4.3] XGBoost
Train XGBoost on the same features and using the same Scaling function

In [0]:
from xgboost import XGBRegressor

In [0]:
scaler = StandardScaler()

In [0]:
xg_model = XGBRegressor(
    n_estimators=1500,        
    learning_rate=0.01,      
    max_depth=3,             
    min_child_weight=15,      
    subsample=0.8,            
    colsample_bytree=0.8,    
    reg_alpha=0.5,            
    reg_lambda=5.0,         
    gamma=0.1,                
    random_state=42,
    tree_method="hist"
)


In [0]:
# Select features + label together in one DataFrame
cols = numerical_features + categorical_features + [label_col]
df_all = df_train.select(*cols)

# One-hot encode categorical features in Spark
df_all_encoded = one_hot_encode_spark(df_all, categorical_features, categories_dict)

# Convert to pandas
df_pd = df_all_encoded.toPandas()

xg_model# Split features and target
X_pd = df_pd.drop(columns=[label_col])
y_pd = df_pd[label_col]

# Scale numerical features
X_pd[numerical_features] = scaler.fit_transform(X_pd[numerical_features])

# Train the model
xg_model.fit(X_pd, y_pd)

In [0]:
 preds = xg_model.predict(X_pd)
 y_true = y_pd
 rmse = np.sqrt(mean_squared_error(y_true, preds))

 print(f'Training set RMSE: {rmse}')

In [0]:
# Select features + label together
cols_val = numerical_features + categorical_features + [label_col]
df_val_all = df_val.select(*cols_val)

# One-hot encode categorical columns
df_val_encoded = one_hot_encode_spark(df_val_all, categorical_features, categories_dict)

# Convert to pandas
df_val_pd = df_val_encoded.toPandas()

# Split features / target
X_val_pd = df_val_pd.drop(columns=[label_col])
y_val_pd = df_val_pd[label_col]

# Scale
X_val_pd[numerical_features] = scaler.transform(X_val_pd[numerical_features])

# Predict
preds_val = xg_model.predict(X_val_pd)
rmse_val = np.sqrt(mean_squared_error(y_val_pd, preds_val))
print(f"Validation set RMSE: {rmse_val:.4f}")

In [0]:
# --- Test RMSE ---
# Prepare test data
X_test = df_test.select(*numerical_features, *categorical_features)
X_test = one_hot_encode_spark(X_test, categorical_features, categories_dict)
X_test = X_test.drop(*categorical_features)

y_test = df_test.select(label_col)

# Convert to Pandas
X_test_pd = X_test.toPandas()
y_test_pd = y_test.toPandas()

# Scale with the SAME scaler (transform only)
X_test_scaled = X_test_pd.copy()
X_test_scaled[numerical_features] = scaler.transform(X_test_pd[numerical_features])

# Predictions + RMSE
preds_test = xg_model.predict(X_test_scaled)
y_true_test = y_test_pd.values.ravel()
rmse_test = np.sqrt(mean_squared_error(y_true_test, preds_test))
print(f"Test set RMSE: {rmse_test:.4f}")

### 5. Test fittnig on entier dataset in monthly Chunks

This part have to be run seperate from the part 3 and 4 to avoid Python Kernal crashing due to memory limit. ohe_spark function need to also be defined

In [0]:
from sklearn.linear_model import SGDRegressor

In [0]:
# Label + features
label_col = "total_amount"
numerical_features = ['trip_distance', 'trip_duration_hours', 'year', 'month']
categorical_features = ['day_of_week', 'hour_of_day']

# --- 1. Build category dictionary globally (once) ---
categories_dict = {}
for col in categorical_features:
    categories_dict[col] = [
        row[col] for row in df_trainval.select(col).distinct().collect()
    ]

# --- 2. Initialize model + scaler ---
reg = SGDRegressor(loss="squared_error", penalty="elasticnet", random_state=42)
scaler = StandardScaler()

# Get sorted year_month chunks
year_months = [row['year_month']
               for row in df_trainval.select("year_month").distinct().collect()]
year_months = sorted(year_months)

first = True
for ym in year_months:
    # --- 3. Filter chunk in Spark ---
    df_chunk = df_trainval.filter(F.col("year_month") == ym)

    # --- 4. One-hot encode categorical cols in Spark ---
    df_chunk = one_hot_encode_spark(df_chunk, categorical_features, categories_dict)

    # --- 5. Select only features + label ---
    feature_cols = numerical_features + [
        c for c in df_chunk.columns if any(c.startswith(cat) for cat in categorical_features)
    ]
    df_chunk = df_chunk.select(feature_cols + [label_col])

    # --- 6. Convert chunk to Pandas ---
    pdf_chunk = df_chunk.toPandas()

    X_chunk = pdf_chunk[feature_cols].values
    y_chunk = pdf_chunk[label_col].values

    # --- 7. Scale + incremental train ---
    if first:
        X_chunk = scaler.fit_transform(X_chunk)
        reg.partial_fit(X_chunk, y_chunk)
        first = False
    else:
        X_chunk = scaler.transform(X_chunk)
        reg.partial_fit(X_chunk, y_chunk)

# --- 8. Final evaluation on test ---
df_test_encoded = one_hot_encode_spark(df_test, categorical_features, categories_dict)
feature_cols = numerical_features + [
    c for c in df_test_encoded.columns if any(c.startswith(cat) for cat in categorical_features)
]
pdf_test = df_test_encoded.select(feature_cols + [label_col]).toPandas()

X_test_scaled = scaler.transform(pdf_test[feature_cols].values)
y_test = pdf_test[label_col].values

y_pred = reg.predict(X_test_scaled)


In [0]:

y_test_pd = df_test.select(label_col).toPandas()[label_col]
rmse_test = np.sqrt(mean_squared_error(y_test_pd, y_pred))
print(f"Test RMSE: {rmse_test:.2f}")
